In [1]:
import nglview as nv
import requests
import pandas as pd
import numpy as np
import io
from Bio.PDB import PDBList, PDBParser
from IPython.display import clear_output
from tqdm import tqdm
from Bio.PDB.SASA import ShrakeRupley

In [10]:
sabdab = '/Users/juliawu/b-cell-epitope/data/sabdab_summary_all.tsv'
df_sabdab = pd.read_csv(sabdab, sep='\t')

df_sabdab['pdb'] = df_sabdab['pdb'].str.upper()
df_sabdab.head()

#df_sabdab[df_sabdab['pdb'] == '9NRX']

,pdb,Hchain,Lchain,model,antigen_chain,antigen_type,antigen_het_name,antigen_name,short_header,date,...,scfv,engineered,heavy_subclass,light_subclass,light_ctype,affinity,delta_g,affinity_method,temperature,pmid
0,10GH,C,B,0,A,protein,NaN,proto-oncogene tyrosine-protein kinase ros,TRANSFERASE/IMMUNE SYSTEM,02/25/26,...,False,True,IGHV3,IGKV1,Kappa,NaN,NaN,NaN,NaN,NaN
1,9DZ4,C,B,0,A,protein,NaN,proto-oncogene tyrosine-protein kinase ros,TRANSFERASE/IMMUNE SYSTEM,02/25/26,...,False,True,IGHV3,IGKV1,Kappa,NaN,NaN,NaN,NaN,NaN
2,9I5N,B,A,0,C,protein,NaN,"cd40 ligand, soluble form",IMMUNE SYSTEM,02/25/26,...,False,True,IGHV2,IGKV17,Kappa,NaN,NaN,NaN,NaN,NaN
3,9J87,E,e,0,B | C,protein | protein,NA | NA,g protein subunit q (gi2-mini-gq chimera) | gu...,MEMBRANE PROTEIN,02/25/26,...,True,True,unknown,unknown,unknown,NaN,NaN,NaN,NaN,NaN
4,9NVG,H,L,0,A,protein,NaN,spike glycoprotein,VIRAL PROTEIN/IMMUNE SYSTEM,02/25/26,...,False,True,IGHV2,IGKV1,Kappa,NaN,NaN,NaN,NaN,NaN


In [3]:
def find_anti_ab_chains(pdb_id):

    Hchain = ''
    Lchain = ''
    Achain = ''

    filtered_rows = df_sabdab[df_sabdab['pdb'] == pdb_id]
    if len(filtered_rows) == 1:
        Hchain += filtered_rows['Hchain']
        Lchain += filtered_rows['Lchain']
        Achain += filtered_rows['antigen_chain']
    elif len(filtered_rows) > 1:
        for i in len(filtered_rows):
            Hchain += filtered_rows.at[i, 'Hchain']
            Lchain += filtered_rows.at[i, 'Lchain']
            Achain += filtered_rows.at[i, 'antigen_chain']
    return Hchain, Lchain, Achain 




# iterate through every chain 
# drop duplicates later in pd 
# can have antibodies binding to dif areas in antigens 
# iterows: index, row --> row parameter, accesses whole row 

   

In [4]:
max_accessibility = {
    'ALA': 106,
    'CYS': 135,
    'ASP': 163,
    'GLU': 194,
    'PHE': 197,
    'GLY': 84,
    'HIS': 184,
    'ILE': 169,
    'LYS': 205,
    'LEU': 164,
    'MET': 188,
    'ASN': 157,
    'PRO': 136,
    'GLN': 198,
    'ARG': 248,
    'SER': 130,
    'THR': 142,
    'VAL': 142,
    'TRP': 227,
    'TYR': 222,
}

backbone_atoms = ['N', 'CA', 'C', 'O', 'H', 'HA', 'HN', 'OXT']
# side chain atoms named by distance from backbone (ex. CB-->  caron beta)

In [5]:
def calculate_epitope(pdb_id, antibody_chains, antigen_chains, cutoff):
        url = f"https://files.rcsb.org/download/{pdb_id}.pdb"
        pdb_text = requests.get(url).text

        parser = PDBParser(QUIET=True)
        structure = parser.get_structure(pdb_id, io.StringIO(pdb_text))
        sr = ShrakeRupley()
        sr.compute(structure, level="R")

        if 0 not in structure.child_dict:
            print(f'you\'re cooked, no model in structure bro')
            return pd.DataFrame(), []
        
        model = structure[0]
        # print(f'pdbid {pdb_id} ab chains {antibody_chains} ag chains {antigen_chains}')


        ag_residues = []
        ag_res_counter = 0
        for chain in antigen_chains:
            if chain not in model:
                print(f'bro antigen chain: {antigen_chains} aint in your model: {model}')
                return pd.DataFrame, []
            for res in model[chain].get_residues():
                if res.id[0] == ' ': # res.id is a tuple (residue, seq number)
                    ag_residues.append(res.resname)
                    ag_res_counter += 1

        if ag_res_counter < 30:
            print(f'antigen is probably truncated! its length is {ag_res_counter}')

        ab_info = []
        ab_atoms = []
        for chain in antibody_chains:
            if chain not in model: 
                print(f'bro antigen chain: {antibody_chains} aint in your model: {model}')
                return pd.DataFrame(), []
            for res in model[chain].get_residues():
                ab_info.append((chain, res.resname))
            for atom in model[chain].get_atoms():
                ab_atoms.append(atom.coord)

        results = []
        residues = []
        for chain in antigen_chains:
            for res in model[chain].get_residues():
                #print(f'residue: {res}')

                # skip water and ligands
                if res.id[0] != " ":
                    continue

                # get all atom coords for this residue
                ag_atoms = []
                #ag_atoms_check = [] # to check if the atoms are in the side chain
                for atom in res.get_atoms():
                    ag_atoms.append(atom.coord) 
                    #ag_atoms_check.append(atom)
                ag_atoms = np.array(ag_atoms)
                #print(ag_atoms_check)


                # calculate distance from each antigen atom to each antibody atom
                for ag_coord in ag_atoms:
                    for ab_coord in ab_atoms:
                        d = np.sqrt(((ag_coord[0] - ab_coord[0])**2)+((ag_coord[1] - ab_coord[1])**2)+((ag_coord[2] - ab_coord[2])**2)) # the euclidean distance cus it's in x,y,z space
                        #print(f'distance: {d}')
                        #print(f'ag_coord: {ag_coord}')
                        #print(f'ab_coord: {ab_coord}')
                        #print(type(ab_coord), ab_coord.shape
                        # if it's under 5A then it's the epitope!!!
                        if d < cutoff:
                            # dupes... multiple atoms under same residue that're under 5 A, but we only want the residue 
                            if res.id[1] not in residues:
                                if atom.get_name() not in backbone_atoms: # (expression for variable in iterable if condition)
                
                                    residues.append(res.id[1])
                                    # print(ag_atoms_check)
                                    
                                    # checking sasa score: if the residue is solvent accessible 
                                    # print(chain)
                                    absolute_sasa = res.get_resname(),round(res.sasa,2) # structure, chain, residue, atom, to 2 decimal places
                                    res_name = str(res.get_resname())
                                    relative_sasa = (absolute_sasa[1] / max_accessibility[res_name]) * 100
                                    
                                    #print(relative_sasa)
                                    if relative_sasa >= 16: #from rost and sander paper

                                        results.append({
                                            "chain"        : chain,
                                            "residue number" : res.id[1],
                                            "residue name" : res_name,
                                            "distance (A)" : round(d, 2)
                                        })
        # print(f'# of epitopes {len(results)}') # comment on what i think things are as i do them 
        df = pd.DataFrame(results)
        if df.empty == True:
            return df, []
        
        resi_list = df['residue number'].to_list()
        string_resi_list = [str(num) for num in resi_list]
        pymol_list = ('+'.join(string_resi_list))

        return(df, pymol_list)

calculate_epitope('10GH', ['B', 'C'], ['A'], 5)

# GET THE ATOM IN THE SIDE CHAIN instead, not in the backbone..... FIX THIS!!!!

# frequencies in EV files 
# a2m file is the homolog pairs, calculate the mutability from the a2m files anyways 
# my work is ground truth --> sequences (ev coupling)does mutability correspond to epitope at all --> using sequences alone 
# does score and mutability correspond to real epitope

(   chain  residue number residue name  distance (A)
 0      A             319          LEU          4.93
 1      A             464          THR          4.87
 2      A             483          THR          4.14
 3      A             484          MET          4.57
 4      A             502          PRO          4.93
 5      A             503          TRP          4.90
 6      A             526          ALA          4.60
 7      A             533          LEU          4.90
 8      A             534          SER          4.17
 9      A             539          ILE          4.36
 10     A             547          ILE          4.84
 11     A             549          GLU          4.43
 12     A             550          PHE          4.23,
 '319+464+483+484+502+503+526+533+534+539+547+549+550')

In [6]:
ls50 = {1:0, 2:0, 3:0, 4:0, 5:0, 'PDB ID': ['5F9W', '5F96', '5F9O', '4RWY', '4RX4']}
df_ls50 = pd.DataFrame(ls50)
print(df_ls50)


   1  2  3  4  5 PDB ID
0  0  0  0  0  0   5F9W
1  0  0  0  0  0   5F96
2  0  0  0  0  0   5F9O
3  0  0  0  0  0   4RWY
4  0  0  0  0  0   4RX4


In [9]:
def parsing_data(insert_df): 
    epitopes = []
    df_new_rows = []
    insert_df = insert_df.copy()

    for i in tqdm(range(len(insert_df))):

        pdb_id = insert_df.iloc[i, 5]

        # have pdb id in who virus--> want to grab rows with same id in sabdab

        sabdab_id = df_sabdab[df_sabdab["pdb"] == pdb_id]

        for x in range(len(sabdab_id)):

            Hchain = sabdab_id.iloc[x,1]
            Lchain = sabdab_id.iloc[x,2]
            Achain = sabdab_id.iloc[x,4] # row contents be a dictionary, key = chain, parse the dictionary instead 

            new_row = insert_df.iloc[i].copy()
            new_row['Hchain'] = Hchain
            new_row['Lchain'] = Lchain
            new_row['Achain'] = Achain

            # check A chain not B | C
            if type(Achain) == float:
                df_new_rows.append(new_row)
                epitopes.append(None)
                continue
            elif len(Achain) > 1:
                split = Achain.split('|')
                multi_A = [i.strip() for i in split]
                for A in multi_A:
                    df, pymol_list = calculate_epitope(pdb_id, [Hchain, Lchain], [A], 5)
                    # print(pymol_list)
                    if type(pymol_list) == str:
                        pymol_list_split = pymol_list.split('+') 
                        prefixed = (f'{A.lower()}{epitope}' for epitope in pymol_list_split)
                        prefixed_plus = ('+'.join(prefixed))
                        epitopes.append(prefixed_plus) # adding which chain its from 
                        df_new_rows.append(new_row) 
                    else:
                        pass
            else: 
                df, pymol_list = calculate_epitope(pdb_id, [Hchain, Lchain], [Achain], 5)
                epitopes.append(pymol_list)
                df_new_rows.append(new_row)
                 

            # eveeytime i append smt to epitopes, i wanna also make a new row 
            
            # print(Hchain, Lchain, Achain)

    
    final_df = pd.DataFrame(df_new_rows)
    print(f'len epitopes list: {len(epitopes)} and insert df: {len(df_new_rows)}')

    final_df['epitope'] = epitopes
    return final_df

final_df = parsing_data(df_ls50)
final_df


# for df, make a list of chains 
# have a row for seq of antigen 

# BROOOOOOOOOOOOOOOOOON FML SO MANY FRICKING KEY ERRORS 5BQZ DOESNT HAVE THE FRICKING CHAIN 

# either add prefixes (c) or do it as a dictionary (KEEP ALL OF IT TOGETHER OR ELSE)

# CROSSCHECK THE PDBS I HAVE WITH THE ONES IN THE WHO GITHUB https://github.com/debbiemarkslab/priority-viruses/tree/main/data/viral_dms_structures
# make sure i include those pdbs 

# EVEscape, EVE, evolutionary coupling 


#LS50 lab:
# bring SASA calculations out
# find pairwise mutation distance 


100%|██████████| 5/5 [01:14<00:00, 14.84s/it]

len epitopes list: 7 and insert df: 7


,1,2,3,4,5,PDB ID,Hchain,Lchain,Achain,epitope
0,0,0,0,0,0,5F9W,B,C,A,102+277+279+281+282+354+357+365+368+425+457+46...
0,0,0,0,0,0,5F9W,H,L,G,97+102+277+279+281+282+365+368+425+460+463+474
1,0,0,0,0,0,5F96,H,L,G,277+279+364+426+430+460+461+462+474
2,0,0,0,0,0,5F9O,H,L,G,276+277+432+460+461+474
3,0,0,0,0,0,4RWY,H,L,A,49+97+122+123+124+198+278+365+366+367+432+460+...
4,0,0,0,0,0,4RX4,H,L,G,97+99+121+275+278+365+366+367+429+432+459+460
4,0,0,0,0,0,4RX4,A,D,E,49+97+99+122+123+278+365+366+367+368+429+432+4...


In [8]:
final_df.sample(1)

,1,2,3,4,5,PDB ID,Hchain,Lchain,Achain,epitope
0,0,0,0,0,0,5F9W,H,L,G,97+102+277+279+281+282+365+368+425+460+463+474


In [ ]:
# Show all rows
pd.set_option('display.max_rows', None)
# Show all columns
pd.set_option('display.max_columns', None)
# Avoid line wraps (show all columns on one line)
pd.set_option('display.width', None)
# Show full content of each cell
pd.set_option('display.max_colwidth', None)

final_df